# Qwen2.5-7B Pretraining Verification

This notebook verifies the pretraining capability of the **Ascend 910B CANN image** by running Qwen2.5-7B pretraining with MindSpeed-LLM.

**Workflow:**
1. Check the environment
2. Prepare the pretraining dataset
3. Clone the MindSpeed-LLM scripts
4. Convert HF weights to Megatron weights
5. Preprocess the data
6. Start pretraining

> Training parameters are set for verification mode (few iterations + short sequences). Increase `TRAIN_ITERS` and `SEQ_LENGTH` for production runs.

## 0. Parameter Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=ImportWarning)
warnings.filterwarnings('ignore', category=UserWarning)

from pathlib import Path

# ===== Path configuration =====
HF_MODEL_DIR = Path('/opt/app-root/src/models/Qwen2.5-7B')
WORK_DIR = Path('/opt/app-root/src/Qwen2.5-7B-work-dir')
MINDSPEED_LLM_DIR = WORK_DIR / 'MindSpeed-LLM'
DATA_DIR = WORK_DIR / 'pretrain_dataset'
OUTPUT_DIR = WORK_DIR / 'output' / 'qwen25_7b_pretrained'
LOGS_DIR = WORK_DIR / 'logs'

# ===== Optional: real dataset path =====
ALPACA_PARQUET = Path('/opt/app-root/src/datasets/alpaca/train-00000-of-00001-a09b74b3ef9c3b56.parquet')

# ===== Ascend environment scripts =====
CANN_ENV = '/usr/local/Ascend/cann/set_env.sh'
ATB_ENV = '/usr/local/Ascend/nnal/atb/set_env.sh'

# ===== Parallel configuration (must match weight conversion) =====
TP = 1   # Tensor parallelism
PP = 4   # Pipeline parallelism; requires at least TP*PP=4 NPUs
# Note: With TP=1, each PP stage has about 1.6-2.2B parameters. The AdamW optimizer states
# (exp_avg + exp_avg_sq, fp32) take about 17.6 GiB, and weights plus gradients exceed the 29 GiB memory on 910B.
# Enable --use-distributed-optimizer (ZeRO-1) during training to shard optimizer states by the DP dimension and reduce memory usage.

# ===== Weight conversion output (path includes parallel config to avoid reusing old weights after TP/PP changes) =====
MCORE_WEIGHTS_DIR = WORK_DIR / 'model_weights' / f'qwen25_mcore_tp{TP}_pp{PP}'

# ===== Training hyperparameters (verification mode) =====
SEQ_LENGTH = 512     # Recommended production value: 4096
TRAIN_ITERS = 50     # Recommended production value: 2000+
MBS = 1
LR = 1.25e-6
MIN_LR = 1.25e-7

# ===== Data preprocessing =====
PROCESSED_DATA_PREFIX = DATA_DIR / 'alpaca'
DATA_PATH = str(DATA_DIR / 'alpaca_text_document')  # preprocess_data.py automatically adds the _text_document suffix

print('Configuration loaded')
print(f'  Model: {HF_MODEL_DIR}')
print(f'  Dataset: {ALPACA_PARQUET}' if ALPACA_PARQUET.exists() else '  Dataset: not found; sample data will be used')
print(f'  TP={TP}, PP={PP}, SEQ={SEQ_LENGTH}, ITERS={TRAIN_ITERS}')

## Helper Functions

In [ ]:
import os
import subprocess

_SUPPRESS_WARNINGS = 'ignore::DeprecationWarning,ignore::ImportWarning,ignore::UserWarning'

def run_cmd(cmd, cwd=None, check=True):
    'Run a bash command in the Ascend environment and stream output in real time'
    env_prefix = f'source {CANN_ENV} && source {ATB_ENV}'
    full_cmd = f'{env_prefix} && {cmd}'
    print(f'$ {cmd}\n')
    run_env = os.environ.copy()
    run_env['PYTHONWARNINGS'] = _SUPPRESS_WARNINGS
    result = subprocess.run(
        ['bash', '-lc', full_cmd],
        cwd=str(cwd or WORK_DIR),
        text=True,
        env=run_env,
    )
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with return code: {result.returncode}')
    return result

print('Helper function defined: run_cmd()')

## 1. Environment Check

In [ ]:
import warnings

with warnings.catch_warnings():
    warnings.simplefilter('ignore', DeprecationWarning)
    warnings.simplefilter('ignore', ImportWarning)
    warnings.simplefilter('ignore', UserWarning)
    import torch
    import torch_npu

print('=' * 60)
print('Environment check')
print('=' * 60)

# PyTorch & NPU
print(f'PyTorch:    {torch.__version__}')
print(f'torch_npu:  {torch_npu.__version__}')
nproc = torch.npu.device_count()
print(f'NPU count:  {nproc}')
for i in range(nproc):
    print(f'  NPU {i}: {torch.npu.get_device_name(i)}')

# MindSpeed
with warnings.catch_warnings():
    warnings.simplefilter('ignore', DeprecationWarning)
    warnings.simplefilter('ignore', ImportWarning)
    warnings.simplefilter('ignore', UserWarning)
    import mindspeed
    import mindspeed_llm

print('MindSpeed:     installed')
print('MindSpeed-LLM: installed')

# Model files
print(f'\nModel directory: {HF_MODEL_DIR}')
assert HF_MODEL_DIR.exists(), f'Model directory does not exist: {HF_MODEL_DIR}'
model_files = sorted(HF_MODEL_DIR.glob('*'))
for f in model_files[:5]:
    if f.is_file():
        print(f'  {f.name} ({f.stat().st_size / 1e9:.2f} GB)')
if len(model_files) > 5:
    print(f'  ... {len(model_files)} files in total')

# Parallel configuration check
assert nproc >= TP * PP, f'NPU count({nproc}) < TP*PP({TP*PP}); reduce PP'
DP = nproc // (TP * PP)
GBS = DP * MBS
print(f'\nParallel configuration: TP={TP}, PP={PP}, DP={DP}, GBS={GBS}')

assert torch.npu.is_available(), 'NPU is not available'
print('\nEnvironment check passed!')

## 2. Prepare the Pretraining Dataset

Pretraining uses raw text data. MindSpeed-LLM's `preprocess_data.py` supports `.parquet`, `.json`, `.jsonl`, `.txt`, and other formats.

This example uses the Alpaca dataset in parquet format, which contains a `text` field. If you use another dataset, make sure it contains a `text` field or uses plain text format.

In [ ]:
import json
import warnings

DATA_DIR.mkdir(parents=True, exist_ok=True)

if ALPACA_PARQUET.exists():
    print(f'Dataset is ready: {ALPACA_PARQUET.name}')
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', DeprecationWarning)
        import pandas as pd
        df = pd.read_parquet(ALPACA_PARQUET)
    print(f'{len(df)} samples, columns: {list(df.columns)}')
    print('\nSample examples:')
    for i, row in df.head(3).iterrows():
        text = str(row.get('text', ''))[:100]
        print(f'  [{i}] {text}...')
    DATA_INPUT = str(ALPACA_PARQUET)
else:
    print('Alpaca dataset not found. Creating sample text data\n')
    sample_texts = [
        {'text': 'Natural language processing is an important branch of artificial intelligence that studies how computers understand and generate human language.'},
        {'text': 'Deep learning uses multilayer neural networks to learn hierarchical data representations and is widely used in computer vision and natural language processing.'},
        {'text': 'Python is a high-level programming language known for its concise, readable syntax and rich ecosystem.'},
        {'text': 'Machine learning is a core artificial intelligence technology that enables computers to learn and improve automatically from data.'},
        {'text': 'Ascend 910B is an artificial intelligence processor from Huawei designed for deep learning training and inference workloads.'},
        {'text': 'Pretraining is the first stage of training large language models and learns statistical patterns from massive text corpora.'},
        {'text': 'The Transformer architecture is a foundation of modern natural language processing and uses self-attention for parallel sequence modeling.'},
        {'text': 'Distributed training spreads large model training workloads across multiple compute devices.'},
        {'text': 'Tensor parallelism and pipeline parallelism are two common parallelism strategies for large model training.'},
        {'text': 'Gradient accumulation simulates large-batch training when memory is limited.'},
    ]
    SAMPLE_FILE = DATA_DIR / 'sample_pretrain.jsonl'
    with open(SAMPLE_FILE, 'w', encoding='utf-8') as f:
        for item in sample_texts:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    DATA_INPUT = str(SAMPLE_FILE)
    print(f'Sample data created: {SAMPLE_FILE}')
    print(f'{len(sample_texts)} samples')

print(f'\nData input path: {DATA_INPUT}')

## 3. Clone MindSpeed-LLM

The `mindspeed_llm` Python package is installed during image build, but the training scripts (`convert_ckpt.py`, `preprocess_data.py`, `pretrain_gpt.py`, and others) must run from the repository directory.

In [ ]:
if MINDSPEED_LLM_DIR.exists():
    print(f'Already exists: {MINDSPEED_LLM_DIR}')
else:
    print('Cloning MindSpeed-LLM (shallow clone)...')
    run_cmd(f'git clone --depth 1 https://gitcode.com/ascend/MindSpeed-LLM.git {MINDSPEED_LLM_DIR}')

# Verify required scripts
scripts = [
    ('Weight conversion', 'convert_ckpt.py'),
    ('Data preprocessing', 'preprocess_data.py'),
    ('Pretraining', 'pretrain_gpt.py'),
]
for name, script in scripts:
    exists = (MINDSPEED_LLM_DIR / script).exists()
    print(f'  [{name}] {script}: {"OK" if exists else "MISSING"}')

assert all((MINDSPEED_LLM_DIR / s).exists() for _, s in scripts), 'Required scripts are missing'
print('\nScript check passed!')

## 4. Convert HF Weights to Megatron Weights

Convert HuggingFace-format weights to Megatron-Mcore format, split by TP/PP. The first conversion usually takes 5-10 minutes.

Qwen2.5 is based on the LLaMA2 architecture with QKV bias, so the conversion uses `--model-type-hf llama2` plus `--add-qkv-bias`.

In [ ]:
MCORE_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# Check whether conversion already exists
converted = any(MCORE_WEIGHTS_DIR.glob('iter_*'))

if converted:
    print(f'Weights already exist; skipping conversion: {MCORE_WEIGHTS_DIR}')
    for p in sorted(MCORE_WEIGHTS_DIR.iterdir()):
        print(f'  {p.name}')
else:
    convert_cmd = ' && '.join([
        f'cd {MINDSPEED_LLM_DIR}',
        'python convert_ckpt.py'
        ' --use-mcore-models'
        ' --model-type GPT'
        ' --load-model-type hf'
        ' --save-model-type mg'
        f' --target-tensor-parallel-size {TP}'
        f' --target-pipeline-parallel-size {PP}'
        ' --add-qkv-bias'
        f' --load-dir {HF_MODEL_DIR}'
        f' --save-dir {MCORE_WEIGHTS_DIR}'
        f' --tokenizer-model {HF_MODEL_DIR / "tokenizer.json"}'
        ' --model-type-hf llama2'
        ' --params-dtype bf16',
    ])
    print('Running weight conversion (about 5-10 minutes)...')
    run_cmd(convert_cmd, cwd=MINDSPEED_LLM_DIR)
    print('Weight conversion completed!')
    for p in sorted(MCORE_WEIGHTS_DIR.iterdir()):
        print(f'  {p.name}')

## 5. Data Preprocessing

Convert text data into the binary format (`.bin` + `.idx`) required by MindSpeed-LLM pretraining.

No handler needs to be specified for pretraining data processing. `preprocess_data.py` automatically extracts the `text` field and generates `alpaca_text_document.bin` and `alpaca_text_document.idx`.

In [ ]:
preprocess_cmd = ' && '.join([
    f'cd {MINDSPEED_LLM_DIR}',
    'python preprocess_data.py'
    f' --input {DATA_INPUT}'
    f' --tokenizer-name-or-path {HF_MODEL_DIR}'
    f' --output-prefix {PROCESSED_DATA_PREFIX}'
    ' --tokenizer-type PretrainedFromHF'
    ' --workers 4'
    ' --log-interval 1000',
])

print('Running data preprocessing...')
run_cmd(preprocess_cmd, cwd=MINDSPEED_LLM_DIR)

# Verify outputs
print('\nPreprocessing outputs:')
for f in sorted(DATA_DIR.glob('alpaca*')):
    print(f'  {f.name} ({f.stat().st_size / 1024:.1f} KB)')

assert (DATA_DIR / 'alpaca_text_document.bin').exists() or (DATA_DIR / 'alpaca_text_document.idx').exists(), \
    f'Preprocessing outputs not found: {DATA_DIR / "alpaca_text_document.*"}'
print('Data preprocessing completed!')

## 6. Start Pretraining

Run Qwen2.5-7B pretraining with MindSpeed-LLM. Training logs are streamed to the notebook in real time.

> In verification mode, `TRAIN_ITERS=50`. For full pretraining, 2000+ iterations are recommended.

**Qwen2.5-7B architecture parameters:**
- 28 Transformer layers, hidden size 3584, FFN size 18944
- 28 attention heads, 4 KV heads (GQA)
- RoPE positional encoding, SwiGLU activation, RMSNorm, QKV bias

In [ ]:
import torch

nproc = torch.npu.device_count()
DP = nproc // (TP * PP)
GBS = DP * MBS

LOGS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Environment variables
env = ' && '.join([
    f'cd {MINDSPEED_LLM_DIR}',
    'export CUDA_DEVICE_MAX_CONNECTIONS=1',
    'export PYTORCH_NPU_ALLOC_CONF=expandable_segments:True',
])

# torchrun distributed arguments
distributed = ' '.join([
    'torchrun',
    f'--nproc_per_node {nproc}',
    '--nnodes 1 --node_rank 0',
    '--master_addr localhost --master_port 6000',
])

# Model architecture (Qwen2.5-7B)
model_args = ' '.join([
    '--use-mcore-models',
    f'--tensor-model-parallel-size {TP}',
    f'--pipeline-model-parallel-size {PP}',
    '--sequence-parallel --use-flash-attn',
    '--transformer-impl local',
    '--use-distributed-optimizer',
    '--num-layers 28 --hidden-size 3584 --num-attention-heads 28',
    '--ffn-hidden-size 18944 --max-position-embeddings 131072',
    f'--seq-length {SEQ_LENGTH}',
    '--make-vocab-size-divisible-by 1 --padded-vocab-size 152064',
    '--position-embedding-type rope --rotary-base 1000000 --use-rotary-position-embeddings',
    '--group-query-attention --num-query-groups 4',
    '--add-qkv-bias --disable-bias-linear',
    '--untie-embeddings-and-output-weights',
    '--swiglu --normalization RMSNorm --norm-epsilon 1e-6',
])

# Training hyperparameters
train_args = ' '.join([
    f'--micro-batch-size {MBS} --global-batch-size {GBS}',
    f'--train-iters {TRAIN_ITERS}',
    '--lr-decay-style cosine --lr-warmup-fraction 0.01',
    '--init-method-std 0.01',
    f'--lr {LR} --min-lr {MIN_LR}',
    '--weight-decay 1e-1 --clip-grad 1.0',
    '--adam-beta1 0.9 --adam-beta2 0.95 --initial-loss-scale 4096',
    '--no-gradient-accumulation-fusion --attention-softmax-in-fp32',
    '--no-masked-softmax-fusion --no-load-optim --no-load-rng',
    '--seed 42 --bf16',
])

# Activation recomputation: recompute forward activations during backward pass to trade compute for memory.
# With PP=4, each stage has 7 layers, so recomputing all layers maximizes memory savings.
recompute_args = ' '.join([
    '--recompute-granularity full',
    '--recompute-method block',
    '--recompute-num-layers 7',
])

# Data and outputs
data_args = ' '.join([
    f'--data-path {DATA_PATH}',
    '--split 100,0,0',
    '--tokenizer-type PretrainedFromHF',
    f'--tokenizer-name-or-path {HF_MODEL_DIR}',
    '--log-interval 1',
    f'--save-interval {TRAIN_ITERS}',
    f'--eval-interval {TRAIN_ITERS} --eval-iters 0',
])

# Load and save
output_args = ' '.join([
    f'--load {MCORE_WEIGHTS_DIR} --save {OUTPUT_DIR}',
    '--distributed-backend nccl',
    '--exit-on-missing-checkpoint',
    '--no-save-optim --no-save-rng',
])

cmd = f'{env} && {distributed} pretrain_gpt.py {model_args} {train_args} {recompute_args} {data_args} {output_args}'

print(f'Training configuration: {nproc} NPU, TP={TP}, PP={PP}, DP={DP}')
print(f'GBS={GBS}, MBS={MBS}, SEQ={SEQ_LENGTH}, ITERS={TRAIN_ITERS}')
print(f'Activation recomputation: full (7 layers per PP stage)')
print(f'\nStarting pretraining...\n')
run_cmd(cmd, cwd=MINDSPEED_LLM_DIR)
print(f'\nPretraining completed! Weights saved to: {OUTPUT_DIR}')

## Use a Real Dataset

After verification succeeds, use a real dataset for full pretraining as follows:

1. **Prepare the data**: place the text dataset inside the container
   - Supported formats: `.parquet`, `.json`, `.jsonl`, `.txt`
   - The data must contain a `text` field (parquet/json/jsonl) or one text segment per line (txt)

2. **Adjust parameters**:
   - `SEQ_LENGTH = 4096` to match the model context length
   - `TRAIN_ITERS = 2000+` depending on dataset size
   - `GBS` based on the NPU count and dataset size; it can be set larger than `DP * MBS` to enable gradient accumulation

3. **Save interval**: modify `--save-interval` in the training cell for periodic checkpoints

4. **Weight conversion**: if TP/PP changes, rerun weight conversion